In [3]:
# --- CELL 1: SETUP & DATA LOADING FUNCTIONS ---
import numpy as np
import pandas as pd
import pyvista as pv
from mpi4py import MPI
import os
import dolfinx
from dolfinx.fem import petsc, Function, functionspace
from dolfinx.mesh import CellType, create_unit_square
import ufl
import basix.ufl

pv.global_theme.allow_empty_mesh = True

# Paths
ODE_RESULTS_PATH = "ode_gof_points_qc.csv" # Ensure this path is correct
OUT_DIR = "results_pde_model"
DATA_PATH = "data/liquidCNA_results/Subclonal_ratio_estimates.extended.txt"
SAMPLE_LIST = "data/OV_patientDNA_sampleList.txt"
os.makedirs(OUT_DIR, exist_ok=True)


In [4]:
# --- CELL 2: DATA CLASSES & LOADING FUNCTIONS ---
from dataclasses import dataclass
from typing import List

@dataclass
class PatientData:
    """Storage for patient clinical data"""
    patient: str
    t: np.ndarray               # time points
    ratio: np.ndarray           # resistant fraction
    se_logit_ratio: np.ndarray  # standard error
    log_ca125: np.ndarray       # tumor size proxy
    ca125: np.ndarray           # raw tumor size

def inverse_logit(x):
    return 1.0 / (1.0 + np.exp(-x))

def load_sample_list(path):
    """Helper to load the sample metadata list"""
    if not path or not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep="\t")
    # Normalize columns
    df.columns = [c.strip() for c in df.columns]
    return df

def load_patient_data(
    path: str,
    patient: str,
    time_unit: str = "months",
    sample_list_path: str = None,
    use_ca125_updated: bool = False,
    drop_failed: bool = False,
    require_panel_sequenced: bool = False,
    require_detected_cna: bool = False,
) -> PatientData:
    """
    Loads and cleans data for a specific patient.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Data file not found: {path}")

    df = pd.read_csv(path, sep="\t")

    # Filter for valid estimates
    if "Accept_estimate" in df.columns:
        df = df[df["Accept_estimate"].isin(["yes", "maybe"])].copy()

    # --- Optional Merge with Sample List (for QC or Updated CA125) ---
    if sample_list_path and os.path.exists(sample_list_path):
        sl = load_sample_list(sample_list_path)

        # Identify sample ID column (usually 'time' in result file vs 'SampleName' in list)
        sample_col = "time" if "time" in df.columns else "SampleName"

        # Merge if possible
        if sl is not None and sample_col in df.columns:
            # Ensure string types for merging
            df[sample_col] = df[sample_col].astype(str)
            sl = sl.rename(columns={"SampleName": sample_col})

            # Merge subset of columns
            cols_to_merge = [sample_col, "Patient"]
            if "CA125_updated" in sl.columns: cols_to_merge.append("CA125_updated")
            if "Failed" in sl.columns: cols_to_merge.append("Failed")

            df = df.merge(sl[cols_to_merge], on=[sample_col, "Patient"], how="left")

            # Apply QC Filters
            if drop_failed and "Failed" in df.columns:
                # Filter out True/ 'True' / 'T'
                is_failed = df["Failed"].astype(str).str.lower().isin(['true', 't', 'yes', '1'])
                df = df[~is_failed]

            # Use Updated CA125 if requested
            if use_ca125_updated and "CA125_updated" in df.columns:
                df["CA125"] = np.where(df["CA125_updated"].notna(), df["CA125_updated"], df["CA125"])

    # --- Cleaning ---
    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
    df["CA125"] = pd.to_numeric(df["CA125"], errors="coerce")

    # Filter for specific patient
    df = df[df["Patient"] == str(patient)].copy()

    # Drop rows with missing critical data
    df = df.dropna(subset=["Time", "ratio", "CA125"])

    if df.empty:
        raise ValueError(f"No valid data found for patient {patient}")

    # Rescale Time
    if time_unit == "months":
        df["Time"] = df["Time"] / 30.0

    df = df.sort_values("Time").reset_index(drop=True)

    # Prepare Arrays
    t = df["Time"].to_numpy().astype(float)
    ratio = np.clip(df["ratio"].to_numpy().astype(float), 1e-4, 1 - 1e-4)
    ca125 = df["CA125"].to_numpy().astype(float)
    log_ca = np.log(np.clip(ca125, 1e-12, None))

    # Calculate approx standard error for weighting (if needed later)
    # Simple placeholder: 10% uncertainty or derived from CIs if available
    se = np.ones_like(ratio) * 0.1

    print(f"   Loaded {len(t)} timepoints for {patient}. Time range: {t.min():.1f} to {t.max():.1f} {time_unit}")

    return PatientData(
        patient=str(patient),
        t=t,
        ratio=ratio,
        se_logit_ratio=se,
        log_ca125=log_ca,
        ca125=ca125
    )

def load_all_patient_params(csv_path):
    """Parses ODE results CSV to get params for ALL patients."""
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Could not find {csv_path}")

    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    val_col = 'pred' if 'pred' in df.columns else 'value'

    params_dict = {}
    for pid in df['patient'].unique():
        sub = df[df['patient'] == pid]

        def get_val(vname):
            row = sub[sub['var'] == vname]
            return float(row[val_col].values[0]) if not row.empty else None

        # Extract Logs/Logits
        log_aS = get_val('theta:log_aS')
        logit_aR = get_val('theta:logit_aR_over_aS')
        log_dS = get_val('theta:log_dS')
        logit_dR = get_val('theta:logit_dR_over_dS')
        log_K = get_val('theta:log_K')

        if any(x is None for x in [log_aS, logit_aR, log_dS, logit_dR, log_K]):
            continue

        # Transform to Physical
        aS = np.exp(log_aS)
        aR = aS * inverse_logit(logit_aR)
        dS = np.exp(log_dS)
        dR = dS * inverse_logit(logit_dR)
        K = np.exp(log_K)

        params_dict[pid] = {"aS": aS, "aR": aR, "dS": dS, "dR": dR, "K": K, "DS": 0.01, "DR": 0.01}

    return params_dict

# Load Params Database
PATIENT_DB = load_all_patient_params(ODE_RESULTS_PATH)
print(f"Params loaded for: {list(PATIENT_DB.keys())}")

Params loaded for: ['UP0018', 'UP0042', 'UP0053', 'UP0055', 'UP0056']


In [5]:
# Stochastic & Parameterized Simulation

# def run_cancer_simulation(params):
#     """
#     Runs the Reaction-Diffusion model with PATIENT SPECIFIC parameters.
#     """
#     comm = MPI.COMM_WORLD
#     msh = create_unit_square(comm, 100, 500, CellType.quadrilateral)
#
#     element = basix.ufl.element("Lagrange", msh.topology.cell_name(), 1)
#     V = functionspace(msh, element)
#
#     S = Function(V); R = Function(V)
#     S_prev = Function(V); R_prev = Function(V)
#
#     # Initial Condition: Scaled by Patient K
#     def init_blob(x):
#         r = np.sqrt((x[0]-0.5)**2 + (x[1]-0.5)**2)
#         return params["K"] * 0.1 * np.exp(-r**2 / 0.05)
#
#     S.interpolate(lambda x: 0.8 * init_blob(x))
#     R.interpolate(lambda x: 0.2 * init_blob(x))
#     S_prev.x.array[:] = S.x.array[:]
#     R_prev.x.array[:] = R.x.array[:]
#
#     dt = 0.5; t_end = 15.0
#     u_trial, v_test = ufl.TrialFunction(V), ufl.TestFunction(V)
#
#     a_S = ufl.inner(u_trial, v_test)*ufl.dx + dt*params["DS"]*ufl.inner(ufl.grad(u_trial), ufl.grad(v_test))*ufl.dx
#     a_R = ufl.inner(u_trial, v_test)*ufl.dx + dt*params["DR"]*ufl.inner(ufl.grad(u_trial), ufl.grad(v_test))*ufl.dx
#     L_S = ufl.inner(S_prev, v_test)*ufl.dx
#     L_R = ufl.inner(R_prev, v_test)*ufl.dx
#
#     # Solver options
#     opts = {"ksp_type": "preonly", "pc_type": "lu"}
#     problem_S = petsc.LinearProblem(a_S, L_S, u=S, petsc_options=opts, petsc_options_prefix="solver_S")
#     problem_R = petsc.LinearProblem(a_R, L_R, u=R, petsc_options=opts, petsc_options_prefix="solver_R")
#
#     # Time Loop
#     t = 0.0
#     while t < t_end:
#         t += dt
#         s_vec = S.x.array; r_vec = R.x.array
#         n_vec = s_vec + r_vec
#
#         growth_s = params["aS"] * s_vec * (1 - n_vec/params["K"])
#         growth_r = params["aR"] * r_vec * (1 - n_vec/params["K"])
#         death_s = 1.0 * params["dS"] * s_vec
#         death_r = 1.0 * params["dR"] * r_vec
#
#         S_prev.x.array[:] = s_vec + dt * (growth_s - death_s)
#         R_prev.x.array[:] = r_vec + dt * (growth_r - death_r)
#
#         problem_S.solve(); problem_R.solve()
#         S.x.array[S.x.array < 0] = 0; R.x.array[R.x.array < 0] = 0
#
#     return msh, S, R

# --- CELL 3: PHYSICS ENGINE (Dynamic Steps) ---
# Exact Simulation
def run_cancer_simulation(params, data):
    """
    Runs the PDE on a 2D Unit Square using PATIENT params and TIME duration.
    """
    comm = MPI.COMM_WORLD
    msh = create_unit_square(comm, 50, 50, CellType.quadrilateral)

    element = basix.ufl.element("Lagrange", msh.topology.cell_name(), 1)
    V = functionspace(msh, element)

    S = Function(V); R = Function(V)
    S_prev = Function(V); R_prev = Function(V)

    # Initial Condition
    def init_blob(x):
        r = np.sqrt((x[0]-0.5)**2 + (x[1]-0.5)**2)
        return params["K"] * 0.1 * np.exp(-r**2 / 0.05)

    S.interpolate(lambda x: 0.8 * init_blob(x))
    R.interpolate(lambda x: 0.2 * init_blob(x))
    S_prev.x.array[:] = S.x.array[:]
    R_prev.x.array[:] = R.x.array[:]

    # [NEW LOGIC] Calculate steps from data
    dt = 0.5
    T_total = float(np.max(data.t))
    num_steps = int(np.ceil(T_total / dt))

    print(f"   Simulating {num_steps} steps (T_max={T_total:.1f} months)...")

    # Forms
    u_trial, v_test = ufl.TrialFunction(V), ufl.TestFunction(V)

    a_S = ufl.inner(u_trial, v_test)*ufl.dx + dt*params["DS"]*ufl.inner(ufl.grad(u_trial), ufl.grad(v_test))*ufl.dx
    a_R = ufl.inner(u_trial, v_test)*ufl.dx + dt*params["DR"]*ufl.inner(ufl.grad(u_trial), ufl.grad(v_test))*ufl.dx
    L_S = ufl.inner(S_prev, v_test)*ufl.dx
    L_R = ufl.inner(R_prev, v_test)*ufl.dx

    opts = {"ksp_type": "preonly", "pc_type": "lu"}
    problem_S = petsc.LinearProblem(a_S, L_S, u=S, petsc_options=opts, petsc_options_prefix="solver_S")
    problem_R = petsc.LinearProblem(a_R, L_R, u=R, petsc_options=opts, petsc_options_prefix="solver_R")

    # [NEW LOGIC] Loop for exact number of steps
    t = 0.0
    for i in range(num_steps):
        t += dt
        s_vec = S.x.array; r_vec = R.x.array
        n_vec = s_vec + r_vec

        growth_s = params["aS"] * s_vec * (1 - n_vec/params["K"])
        growth_r = params["aR"] * r_vec * (1 - n_vec/params["K"])
        death_s = 1.0 * params["dS"] * s_vec
        death_r = 1.0 * params["dR"] * r_vec

        S_prev.x.array[:] = s_vec + dt * (growth_s - death_s)
        R_prev.x.array[:] = r_vec + dt * (growth_r - death_r)

        problem_S.solve()
        problem_R.solve()

        S.x.array[S.x.array < 0] = 0
        R.x.array[R.x.array < 0] = 0

    return msh, S, R

In [6]:
def plot_meshtags_3d(msh, S, R, pid, out_dir=OUT_DIR):
    """
    3D Resistance Zones (Saved to PNG)
    """
    s_vals = S.x.array
    r_vals = R.x.array

    # 1. Calculate Density and Fraction
    total_density = s_vals + r_vals
    ratio = np.divide(r_vals, total_density, out=np.zeros_like(r_vals), where=total_density > 1e-9)

    # 2. Setup Grid
    cells, types, x = dolfinx.plot.vtk_mesh(msh)
    grid = pv.UnstructuredGrid(cells, types, x)
    grid.point_data["Resistant Fraction"] = ratio
    grid.point_data["Density"] = total_density

    # [CHANGE 1 & 2] Set off_screen=True and define window size (width, height)
    pl = pv.Plotter(off_screen=True, window_size=[1024, 768])
    pl.add_text(f"{pid}: Resistance Map", font_size=12)

    # 3. Intelligent Thresholding
    max_dens = np.max(total_density)

    if max_dens < 1e-3:
        pl.add_text(" (Tumor too small)", font_size=10, position="upper_left")
        pl.add_mesh(grid, style="wireframe", color="grey", opacity=0.3)
    else:
        tumor_body = grid.threshold(max_dens * 0.01, scalars="Density")
        pl.add_mesh(tumor_body, scalars="Resistant Fraction", cmap="inferno", clim=[0, 1], show_scalar_bar=True)
        pl.add_mesh(grid, style="wireframe", color="grey", opacity=0.1)

    # [CHANGE 3] Save screenshot and close to free memory
    filepath = os.path.join(out_dir, f"{pid}_resistance_zones.png")
    pl.screenshot(filepath)
    pl.close()
    print(f"   Saved: {filepath}")

def plot_streamlines_3d(msh, S, R, pid, out_dir=OUT_DIR):
    """Auto-scaling 3D Streamlines (Saved to PNG)"""
    V = S.function_space
    N = Function(V)
    N.x.array[:] = S.x.array[:] + R.x.array[:]

    cells, types, x = dolfinx.plot.vtk_mesh(V)
    grid = pv.UnstructuredGrid(cells, types, x)
    grid.point_data["Density"] = N.x.array

    # 1. Find Tumor Edge
    max_dens = np.max(N.x.array)
    if max_dens < 1e-3:
        print(f"   [Viz Warning] {pid} tumor too small for streamlines.")
        return

    tumor_vol = grid.threshold(max_dens * 0.1)
    if tumor_vol.n_points == 0:
        radius = 0.1
    else:
        bounds = tumor_vol.bounds
        width = bounds[1] - bounds[0]
        radius = width / 3.0

    grad = grid.compute_derivative(scalars="Density")
    grad.set_active_vectors("gradient")

    # 2. Seed Points
    stream = grad.streamlines(
        "gradient",
        source_center=(0.5,0.5,0.0),
        source_radius=radius,
        n_points=150,
        max_time=10.0,
        integration_direction="both"
    )

    # [CHANGE 1 & 2]
    pl = pv.Plotter(off_screen=True, window_size=[1024, 768])
    pl.add_text(f"{pid}: Growth Streamlines", font_size=12)
    pl.add_mesh(grid, style="wireframe", opacity=0.1, color="grey")

    if stream.n_points > 0:
        pl.add_mesh(stream.tube(radius=0.003), color="red")

    if grad.n_points > 0:
        arrows = grad.glyph(orient="gradient", scale="gradient", factor=0.1, tolerance=0.01)
        pl.add_mesh(arrows, color="blue", opacity=0.6)

    # [CHANGE 3]
    filepath = os.path.join(out_dir, f"{pid}_streamlines.png")
    pl.screenshot(filepath)
    pl.close()
    print(f"   Saved: {filepath}")

def plot_drug_efficacy(msh, S, R, pid, params, out_dir=OUT_DIR):
    """Plot 5: Drug Kill Map (Saved to PNG)"""
    V = S.function_space # Fixed missing V definition
    s_vals = S.x.array
    kill_rate = params["dS"] * s_vals

    cells, types, x = dolfinx.plot.vtk_mesh(V)
    grid = pv.UnstructuredGrid(cells, types, x)

    grid.point_data["Density"] = S.x.array[:] + R.x.array[:]
    grid.point_data["Kill Rate"] = kill_rate

    centers = grid.points
    vectors = centers - [0.5, 0.5, 0]
    grid["Vectors"] = vectors

    # Fixed arrow object issue
    arrows = grid.glyph(orient="Vectors", scale="Density", geom=pv.Arrow(), factor=0.1, tolerance=0.02)

    # [CHANGE 1 & 2]
    pl = pv.Plotter(off_screen=True, window_size=[1024, 768])
    pl.add_text(f"{pid}: Drug Efficacy (Red = High Kill)", font_size=12)

    if arrows.n_points > 0:
        pl.add_mesh(arrows, scalars="Kill Rate", cmap="coolwarm", show_scalar_bar=True)

    pl.add_mesh(grid, style="wireframe", color="grey", opacity=0.1)

    # [CHANGE 3]
    filepath = os.path.join(out_dir, f"{pid}_drug_efficacy.png")
    pl.screenshot(filepath)
    pl.close()
    print(f"   Saved: {filepath}")

In [7]:
# --- CELL 4: MAIN EXECUTION LOOP ---

# Loop through every patient in the database
for pid, params in PATIENT_DB.items():
    print(f"==========================================")
    print(f"PROCESSING PATIENT: {pid}")
    print(f"Params: aS={params['aS']:.3f}, K={params['K']:.0f}")
    print(f"==========================================")

    # 1. Run Physics
    # mesh_obj, S_field, R_field = run_cancer_simulation(params)

    data = load_patient_data(
                DATA_PATH, pid,
                sample_list_path=SAMPLE_LIST,
                use_ca125_updated=True
            )
    mesh_obj, S_field, R_field = run_cancer_simulation(params, data)
    # 2. Show Separate Plots
    plot_meshtags_3d(mesh_obj, S_field, R_field, pid)
    plot_streamlines_3d(mesh_obj, S_field, R_field, pid)
    plot_drug_efficacy(mesh_obj, S_field, R_field, pid, params)

PROCESSING PATIENT: UP0018
Params: aS=0.285, K=152836
   Loaded 3 timepoints for UP0018. Time range: 0.0 to 23.3 months
   Simulating 47 steps (T_max=23.3 months)...


2025-12-27 03:19:25.531 (  48.289s) [    73E508283740]vtkXOpenGLRenderWindow.:1458  WARN| bad X server connection. DISPLAY=


   Saved: results_pde_model/UP0018_resistance_zones.png


/home/abhinav/miniconda3/envs/cna-resist-dynamics/lib/python3.11/site-packages/pyvista/core/filters/data_set.py:3204: PyVistaDeprecationWarning: ``max_time`` parameter is deprecated.  It will be removed in v0.48
  warnings.warn(


   Saved: results_pde_model/UP0018_streamlines.png
   Saved: results_pde_model/UP0018_drug_efficacy.png
PROCESSING PATIENT: UP0042
Params: aS=0.575, K=1161081
   Loaded 2 timepoints for UP0042. Time range: 0.0 to 12.6 months
   Simulating 26 steps (T_max=12.6 months)...
   Saved: results_pde_model/UP0042_resistance_zones.png


...

   Saved: results_pde_model/UP0053_streamlines.png
   Saved: results_pde_model/UP0053_drug_efficacy.png
PROCESSING PATIENT: UP0055
Params: aS=0.434, K=931805
   Loaded 4 timepoints for UP0055. Time range: 0.0 to 36.8 months
   Simulating 74 steps (T_max=36.8 months)...
   Saved: results_pde_model/UP0055_resistance_zones.png


/home/abhinav/miniconda3/envs/cna-resist-dynamics/lib/python3.11/site-packages/pyvista/core/filters/data_set.py:3204: PyVistaDeprecationWarning: ``max_time`` parameter is deprecated.  It will be removed in v0.48
  warnings.warn(


   Saved: results_pde_model/UP0055_streamlines.png
   Saved: results_pde_model/UP0055_drug_efficacy.png
PROCESSING PATIENT: UP0056
Params: aS=0.073, K=324529
   Loaded 2 timepoints for UP0056. Time range: 0.2 to 8.2 months
   Simulating 17 steps (T_max=8.2 months)...
   Saved: results_pde_model/UP0056_resistance_zones.png


/home/abhinav/miniconda3/envs/cna-resist-dynamics/lib/python3.11/site-packages/pyvista/core/filters/data_set.py:3204: PyVistaDeprecationWarning: ``max_time`` parameter is deprecated.  It will be removed in v0.48
  warnings.warn(


   Saved: results_pde_model/UP0056_streamlines.png
   Saved: results_pde_model/UP0056_drug_efficacy.png
